# microstructure-mri API Walkthrough

This notebook is a practical syntax guide for users who are new to the codebase.

It demonstrates:
- acquisition constructors and presets
- model composition from compartments
- spherical mean simulation
- SANDI and NODDI presets
- least-squares and MCMC fitting syntax
- quadrature and hypergeometric ratio signal pathways

In [1]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/.")

repo_root = find_repo_root(Path.cwd().resolve())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np

from dmri_models import (
    Acquisition,
    Ball,
    Component,
    EmceeFitter,
    GaussianNoise,
    LeastSquaresFitter,
    Model,
    Parameter,
    ParameterSpace,
    Simplex,
    Stick,
    Zeppelin,
    noddi,
    noddi_derived_parameters,
    odi_from_kappa,
    sandi,
)
from dmri_models.acquisitions import (
    bigmac_highres,
    bigmac_lowres,
    hcp,
    names as acquisition_names,
    sandi_full_acq,
    uk_biobank,
)
from dmri_models.compartments import (
    BinghamFOD,
    BinghamStickHypergeometric,
    FODConvolved,
    Sphere,
    WatsonFOD,
    WatsonStickHypergeometric,
    WatsonZeppelinHypergeometric,
)

rng = np.random.default_rng(7)

## 1) Acquisition setup used in this notebook

For consistency, all general examples below use the HCP acquisition.

For SANDI-specific examples, we use sandi_full_acq.

In [5]:
acq_hcp = hcp()
acq_sandi = sandi_full_acq()

print("HCP acquisition summary:")
print(acq_hcp.summary())
print()
print('----------------------------------------------')
print("SANDI acquisition summary:")
print(acq_sandi.summary())

HCP acquisition summary:
Acquisition summary
- measurements: 288
- b-value shells (ms/um^2): 4 (tolerance: 0.100)
  - b=0.000: 18 measurements, 18 non-zero directions
  - b=0.998: 90 measurements, 90 non-zero directions
  - b=1.998: 90 measurements, 90 non-zero directions
  - b=2.996: 90 measurements, 90 non-zero directions
- gradient directions: 288 unique non-zero across 288 diffusion-weighted measurements
- b-tensor shape: 1.000
- pulse duration (ms): 10.600
- diffusion time (ms): 43.100
- gradient strength (T/m): 0.000 to 0.097
- echo time (ms): 89.500
- inversion time: n/a
- repetition time (ms): 5520.000

----------------------------------------------
SANDI acquisition summary:
Acquisition summary
- measurements: 170
- b-value shells (ms/um^2): 6 (tolerance: 0.100)
  - b=0.000: 20 measurements, 0 non-zero directions
  - b=0.500: 30 measurements, 30 non-zero directions
  - b=1.000: 30 measurements, 30 non-zero directions
  - b=2.000: 30 measurements, 30 non-zero directions
  - b=3

## 2) Packaged acquisitions

In [7]:
print("Available acquisition presets:", acquisition_names())

preset_map = {
    "hcp": hcp(),
    "uk_biobank": uk_biobank(),
    "bigmac_highres": bigmac_highres(),
    "bigmac_lowres": bigmac_lowres(),
    "sandi_full_acq": sandi_full_acq(),
}

for name, acq in preset_map.items():
    print("-", name, "->", acq.n_measurements, "measurements")

Available acquisition presets: ('hcp', 'uk_biobank', 'bigmac_highres', 'bigmac_lowres', 'sandi_full_acq')
- hcp -> 288 measurements
- uk_biobank -> 105 measurements
- bigmac_highres -> 136 measurements
- bigmac_lowres -> 260 measurements
- sandi_full_acq -> 170 measurements


## 3) Build a custom model from compartments

This example uses a stick plus isotropic ball.

In [8]:
custom_model = Model([
    Component("neurite", Stick(diffusivity="d_in", orientation="axis")),
    Component("extra", Ball(diffusivity="d_ex")),
])

custom_params = {
    "S0": 1.0,
    "neurite_fraction": 0.65,
    "extra_fraction": 0.35,
    "d_in": 1.8,
    "d_ex": 0.9,
    "axis": np.array([0.4, 0.2, 0.9]),
}

custom_signal = custom_model.signal(acq_hcp, custom_params)
print("Signal length on HCP:", custom_signal.shape[0])
print("First 10 points:", np.round(custom_signal[:10], 4))

Signal length on HCP: 288
First 10 points: [1.     0.766  0.2202 0.0504 0.7809 0.4047 0.1921 0.2597 0.4071 0.1419]


## 4) Spherical mean API

Use `model.spherical_mean_signal(...)` for robust model-level powder averages.

In [18]:
mean_signal = custom_model.spherical_mean_signal(
    acquisition=acq_hcp,
    parameters=custom_params,
    n_directions=256,
    rng=rng,
)
print("Spherical mean signal length:", mean_signal.shape[0])
print("First 10 points:", np.round(mean_signal[:10], 4))

Spherical mean signal length: 288
First 10 points: [1.     0.5469 0.3599 0.2709 0.5482 0.2715 0.3586 0.5496 0.3605 0.2712]


## 5) Preset models: SANDI and NODDI

SANDI preset options:
- `vary_sphere_diffusivity=True/False`

NODDI preset options:
- `fod_model='watson' or 'bingham'`
- `include_csf=True/False`
- `vary_intracellular_diffusivity=True/False`
- `vary_csf_diffusivity=True/False`
- `vary_kappa=True/False`

In this notebook:
- general examples use `acq_hcp`
- SANDI examples use `acq_sandi`

In [11]:
sandi_fixed = sandi(vary_sphere_diffusivity=False)
sandi_vary = sandi(vary_sphere_diffusivity=True)

noddi_watson = noddi(fod_model="watson", include_csf=True)
noddi_bingham = noddi(fod_model="bingham", include_csf=False, vary_kappa=True)

print("SANDI varying parameters (fixed d_sphere):", sandi_fixed.parameters.varying_names)
print("SANDI varying parameters (free d_sphere):", sandi_vary.parameters.varying_names)
print("NODDI Watson varying parameters:", noddi_watson.parameters.varying_names)
print("NODDI Bingham varying parameters:", noddi_bingham.parameters.varying_names)

sandi_signal = sandi_fixed.model.signal(acq_sandi, sandi_fixed.parameters.initial_values())
print("SANDI signal length on SANDI acquisition:", sandi_signal.shape[0])
print("First 10 SANDI points:", np.round(sandi_signal[:10], 4))

derived = noddi_derived_parameters({"kappa": 2.0, "neurite_fraction": 0.55, "csf_fraction": 0.10})
print("Example ODI from utility:", odi_from_kappa(2.0))
print("Derived parameters:", derived)

SANDI varying parameters (fixed d_sphere): ('S0', 'r_soma', 'soma_fraction/extra_fraction', 'neurite_fraction/extra_fraction')
SANDI varying parameters (free d_sphere): ('S0', 'r_soma', 'd_sphere', 'soma_fraction/extra_fraction', 'neurite_fraction/extra_fraction')
NODDI Watson varying parameters: ('S0', 'theta', 'phi', 'kappa', 'neurite_fraction/csf_fraction', 'extra_fraction/csf_fraction')
NODDI Bingham varying parameters: ('S0', 'theta', 'phi', 'kappa1', 'kappa2', 'psi', 'neurite_fraction/extra_fraction')
SANDI signal length on SANDI acquisition: 170
First 10 SANDI points: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Example ODI from utility: 0.2951672353008665
Derived parameters: {'odi': 0.2951672353008665, 'v_ic_tissue': 0.6111111111111112}


## 6) Least-squares fitting syntax (single signal)

In [12]:
fit_model = Model([
    Component("neurite", Stick(diffusivity="d_in", powder_average=True)),
    Component("extra", Ball(diffusivity="d_ex")),
])

fit_space = ParameterSpace(
    [
        Parameter("S0", 1.0, (0.5, 1.5)),
        Parameter("d_in", 1.5, (0.1, 3.0)),
        Parameter("d_ex", 0.8, (0.1, 3.0)),
    ],
    simplexes=[Simplex(("neurite_fraction", "extra_fraction"), (0.6, 0.4))],
)

truth = {
    "S0": 1.0,
    "d_in": 1.9,
    "d_ex": 0.7,
    "neurite_fraction": 0.7,
    "extra_fraction": 0.3,
}

signal_true = fit_model.signal(acq_hcp, truth)
signal_noisy = signal_true + rng.normal(0.0, 0.01, size=signal_true.shape)

fitter = LeastSquaresFitter(fit_model, fit_space)
fit = fitter.fit(acq_hcp, signal_noisy, sigma=0.01, n_starts=12, max_nfev=2000, rng=rng)
print("Fit success:", fit.success, "cost:", round(fit.cost, 6), "starts:", fit.n_starts)
print(fit.parameters)

Fit success: True cost: 122.691008 starts: 12
{'S0': 0.9986570626144303, 'd_in': 1.7579401296916866, 'd_ex': 0.7737249372531773, 'neurite_fraction': 0.6873183193040202, 'extra_fraction': 0.31268168069597985}


## 7) Least-squares fitting syntax (batch)

In [15]:
batch_data = np.stack([
    signal_true + rng.normal(0.0, 0.01, size=signal_true.shape),
    signal_true + rng.normal(0.0, 0.01, size=signal_true.shape),
    signal_true + rng.normal(0.0, 0.01, size=signal_true.shape),
])

batch_result = fitter.fit_many(
    acq_hcp,
    batch_data,
    sigma=0.01,
    n_starts=6,
    n_jobs=1,
    rng=rng,
)

print("Batch success flags:", batch_result.success)
print("Batch costs:", np.round(batch_result.cost, 6))
print("Batch parameters:", batch_result.parameters)

Batch success flags: [ True  True  True]
Batch costs: [176.10981  137.106409 141.552626]
Batch parameters: {'S0': array([0.99717969, 0.99862402, 1.00108208]), 'd_in': array([1.92331947, 1.96359829, 2.13799758]), 'd_ex': array([0.67971821, 0.64886154, 0.58646635]), 'neurite_fraction': array([0.70337552, 0.68591009, 0.69535028]), 'extra_fraction': array([0.29662448, 0.31408991, 0.30464972])}


## 8) MCMC syntax

This cell runs a short chain for demonstration. Increase steps and walkers for real analyses.

In [16]:
try:
    mcmc = EmceeFitter(fit_model, fit_space, GaussianNoise(sigma=0.01))
    posterior = mcmc.sample(
        acq_hcp,
        signal_noisy,
        n_steps=120,
        burn_in=30,
        thin=2,
        n_walkers=24,
        rng=rng,
        progress=False,
    )
    print("Posterior sample shape:", posterior.latent_samples.shape)
    print("Acceptance fraction mean:", float(np.mean(posterior.acceptance_fraction)))
    print("Summary for S0:", posterior.summary()["S0"])
except ImportError as exc:
    print("MCMC demo skipped (optional dependency not installed):", exc)

Posterior sample shape: (45, 24, 4)
Acceptance fraction mean: 0.296875
Summary for S0: {'mean': 0.9991637551990049, 'sd': 0.06783576953574204, 'median': 1.0014879336990115, 'lower': 0.7056750012209814, 'upper': 1.1050540672246012}


For MCMC, I would look at:

summaries for all parameters
pairwise scatter plots or corner plot
autocorrelation time
effective sample size
trace plots for a few walkers

## 9) Quadrature vs hypergeometric ratio: Watson and Bingham stick

The package supports both pathways and they should agree numerically.

In [23]:
acq_oriented = acq_hcp

watson_params = {"d": 1.6, "kappa": 3.0, "theta": 1.0, "phi": -0.4}

watson_quad = FODConvolved(
    response=Stick(diffusivity="d", orientation="axis"),
    fod=WatsonFOD(kappa="kappa", theta="theta", phi="phi", n_theta=12, n_phi=24),
    orientation_parameter="axis",
).signal(acq_oriented, watson_params)

watson_hyp = WatsonStickHypergeometric(
    diffusivity="d",
    kappa="kappa",
    theta="theta",
    phi="phi",
    n_theta=12,
    n_phi=24,
).signal(acq_oriented, watson_params)

print("Watson stick max abs difference:", float(np.max(np.abs(watson_quad - watson_hyp))))

bingham_params = {
    "d": 1.7,
    "kappa1": -8.0,
    "kappa2": -2.0,
    "theta": 1.2,
    "phi": 0.6,
    "psi": -0.3,
}

bingham_quad = FODConvolved(
    response=Stick(diffusivity="d", orientation="axis"),
    fod=BinghamFOD(
        kappa1="kappa1",
        kappa2="kappa2",
        theta="theta",
        phi="phi",
        psi="psi",
        n_theta=12,
        n_phi=24,
    ),
    orientation_parameter="axis",
).signal(acq_oriented, bingham_params)

bingham_hyp = BinghamStickHypergeometric(
    diffusivity="d",
    kappa1="kappa1",
    kappa2="kappa2",
    theta="theta",
    phi="phi",
    psi="psi",
    n_theta=12,
    n_phi=24,
).signal(acq_oriented, bingham_params)

print("Bingham stick max abs difference:", float(np.max(np.abs(bingham_quad - bingham_hyp))))

Watson stick max abs difference: 7.771561172376096e-16
Bingham stick max abs difference: 1.1102230246251565e-15


## 10) Watson + Zeppelin via hypergeometric ratio

For zeppelin responses the matrix argument uses `delta_d = d_parallel - d_perpendicular`, rather than `d_parallel` directly.

In [24]:
watson_zeppelin_params = {
    "d_parallel": 1.8,
    "d_perpendicular": 0.5,
    "kappa": 4.0,
    "theta": 1.0,
    "phi": -0.2,
}

watson_zeppelin_quad = FODConvolved(
    response=Zeppelin(
        parallel_diffusivity="d_parallel",
        perpendicular_diffusivity="d_perpendicular",
        orientation="axis",
    ),
    fod=WatsonFOD(kappa="kappa", theta="theta", phi="phi", n_theta=12, n_phi=24),
    orientation_parameter="axis",
).signal(acq_hcp, watson_zeppelin_params)

watson_zeppelin_hyp = WatsonZeppelinHypergeometric(
    parallel_diffusivity="d_parallel",
    perpendicular_diffusivity="d_perpendicular",
    kappa="kappa",
    theta="theta",
    phi="phi",
    n_theta=12,
    n_phi=24,
).signal(acq_hcp, watson_zeppelin_params)

print("Watson zeppelin max abs difference:", float(np.max(np.abs(watson_zeppelin_quad - watson_zeppelin_hyp))))

Watson zeppelin max abs difference: 8.881784197001252e-16


## Next steps

- Start with SANDI or NODDI presets and inspect `preset.parameters.initial_values()`.
- Use `n_starts` for robust least-squares fitting.
- Increase MCMC chain length for inference, then inspect ESS/autocorrelation.
- For Ball-and-Rackets style studies, compare quadrature and hypergeometric pathways as a regression check.